# ELMo
ELMo模型的从零实现

In [1]:
# 导入与设置（Code）
import torch  # PyTorch 主库：张量计算与自动求导
import torch.nn as nn  # 神经网络模块（层、损失等）
import torch.nn.functional as F  # 函数式 API（激活函数、loss 等）
from torch.utils.data import Dataset, DataLoader  # 数据集抽象与批加载器
from datasets import load_dataset  # HuggingFace datasets：下载/读取公开数据集
from collections import Counter  # 词频统计工具
import re  # 正则表达式：用于基础分词
import numpy as np  # 数值计算与随机数
from tqdm import tqdm  # 训练进度条
import os  # 操作系统接口（路径、文件等）
import math  # 数学函数（如 exp）

# 自动选择运行设备：有 CUDA 就用 GPU，否则用 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# 固定随机种子，确保可复现实验结果（在同环境下尽量一致）
torch.manual_seed(42)
np.random.seed(42)

Device: cuda


## 1) 数据与词表准备
这一部分会从 WikiText-2 数据集中读取文本，并做基础分词清洗。
随后分别构建：
- 词级词表（用于语言模型预测词ID）
- 字符级词表（用于 CharCNN 处理 OOV 和形态信息）
这是 ELMo 中“字符建词向量”的基础输入。

In [2]:
# 加载数据集 & 构建 vocab（Code）
# 读取 WikiText-2 原始文本版本（包含 train / validation / test）
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# 定义清洗 + 分词函数：输入为若干文本行，输出为 token 序列列表
def clean_and_tokenize(lines):
    sentences = []  # 用于保存每一行对应的 token 列表
    for line in lines:
        line = line.strip()  # 去掉行首尾空白
        # 条件1：非空行；条件2：过滤形如 " = Title = " 的标题行
        if line and not line.startswith(" = "):
            # 使用正则分词：\w+ 匹配单词；[^\w\s] 匹配标点
            tokens = re.findall(r"\w+|[^\w\s]", line.lower())  # 统一小写
            if tokens:  # 仅保留非空 token 结果
                sentences.append(tokens)
    return sentences

# 分别处理训练集、验证集、测试集文本
train_sents = clean_and_tokenize(dataset["train"]["text"])
valid_sents = clean_and_tokenize(dataset["validation"]["text"])
test_sents  = clean_and_tokenize(dataset["test"]["text"])

# 打印训练句子数量，便于检查数据规模
print(f"Train sentences: {len(train_sents)}")

# -------------------------
# 1) 构建词级词表（Word Vocab）
# -------------------------
# 将训练集中的所有词打平成一个大列表
all_words = [w for sent in train_sents for w in sent]
# 统计每个词出现次数
word_counter = Counter(all_words)
# 词表前两项保留给 PAD 与 UNK，其余取高频词（总上限约 30k）
word_vocab = ["<PAD>", "<UNK>"] + [w for w, c in word_counter.most_common(29998)]
# 词到索引映射
word2idx = {w: i for i, w in enumerate(word_vocab)}
# 索引到词映射（便于调试和可视化）
idx2word = {i: w for w, i in word2idx.items()}
# 词表大小
VOCAB_SIZE = len(word_vocab)
print("Word vocab size:", VOCAB_SIZE)

# -------------------------
# 2) 构建字符级词表（Char Vocab）
# -------------------------
# 从 train/valid/test 所有句子中收集出现过的字符集合
all_chars = set("".join("".join(sent) for sent in train_sents + valid_sents + test_sents))
# 字符词表同样预留 PAD 与 UNK
char_vocab = ["<PAD>", "<UNK>"] + sorted(list(all_chars))
# 字符到索引映射
char2idx = {c: i for i, c in enumerate(char_vocab)}
# 索引到字符映射
idx2char = {i: c for c, i in char2idx.items()}
# 字符词表大小
CHAR_VOCAB_SIZE = len(char_vocab)
print("Char vocab size:", CHAR_VOCAB_SIZE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train sentences: 23767
Word vocab size: 30000
Char vocab size: 1080


## 2) Dataset 与批处理对齐
这一部分把句子样本封装成 `Dataset`，并在 `collate_fn` 中做动态 padding。
输出会包含：
- `word_ids`: 形状 `(B, S)`
- `char_ids`: 形状 `(B, S, C)`
后续 CharCNN 与 BiLM 会直接消费这两类张量。

In [3]:
# Dataset & Collate（Code）
MAX_WORD_LEN = 20   # 每个词最多保留多少个字符（超长截断）
MAX_SEQ_LEN = 35    # 每个句子最多保留多少个词（超长截断）

# 自定义 ELMo 数据集：把分词后的句子转成 word_id 与 char_id
class ELMoDataset(Dataset):
    # 初始化：传入句子列表、词表映射、字符映射
    def __init__(self, sentences, word2idx, char2idx):
        self.sentences = sentences
        self.word2idx = word2idx
        self.char2idx = char2idx
    
    # 返回样本数（DataLoader 会依赖这个函数）
    def __len__(self):
        return len(self.sentences)
    
    # 获取第 idx 条样本，并编码成张量
    def __getitem__(self, idx):
        # 先截断句子长度，避免单条样本过长占用过多显存
        sent = self.sentences[idx][:MAX_SEQ_LEN]

        # 词级编码：词 -> word_id，未登录词映射到 <UNK>
        word_ids = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in sent]

        # 字符级编码：词 -> 字符序列 -> char_id 序列
        char_ids = []
        for w in sent:
            # 对每个词截取前 MAX_WORD_LEN 个字符并映射到 id
            chs = [self.char2idx.get(c, self.char2idx["<UNK>"]) for c in w[:MAX_WORD_LEN]]
            # 不足 MAX_WORD_LEN 的部分用 <PAD> 补齐
            chs += [self.char2idx["<PAD>"]] * (MAX_WORD_LEN - len(chs))
            char_ids.append(chs)

        # 返回两个张量：
        # word_ids: (seq_len)
        # char_ids: (seq_len, max_word_len)
        return torch.tensor(word_ids), torch.tensor(char_ids)

# 自定义批处理函数：将不同长度句子做 batch 内动态 padding
def collate_fn(batch):
    # batch 是一个列表，每项为 (word_tensor, char_tensor)
    word_ids = [b[0] for b in batch]
    char_ids = [b[1] for b in batch]
    
    # 词级序列 padding：补到本 batch 的最大句长，PAD 值为 0
    word_padded = torch.nn.utils.rnn.pad_sequence(word_ids, batch_first=True, padding_value=0)
    # 字符级序列 padding：同样按句长补齐，字符位本身已在 __getitem__ 中补齐
    char_padded = torch.nn.utils.rnn.pad_sequence(char_ids, batch_first=True, padding_value=0)
    
    # 返回形状：
    # word_padded: (B, S)
    # char_padded: (B, S, C)
    return word_padded, char_padded

## 3) Highway 网络
Highway 可以理解为“带门控的残差通道”。
它让网络学会在“保留原始特征”和“使用非线性变换特征”之间自适应选择，
这在 CharCNN 输出后非常常见，可提升表示稳定性。

In [4]:
# Highway Network（Code）
# Highway 层：通过门控机制在“变换后的特征”和“原始特征”之间做加权融合
class Highway(nn.Module):
    # dim: 输入/输出维度；num_layers: 堆叠多少层 Highway
    def __init__(self, dim, num_layers=2):
        super().__init__()
        # 每层线性层输出 2*dim：前半用于 gate，后半用于 transform
        self.layers = nn.ModuleList([nn.Linear(dim, dim * 2) for _ in range(num_layers)])
    
    # x 形状通常为 (N, dim)
    def forward(self, x):
        for layer in self.layers:
            # 将线性输出沿最后一维一分为二
            gate, transform = layer(x).chunk(2, dim=-1)
            # gate 经过 sigmoid，约束在 [0,1]
            gate = torch.sigmoid(gate)
            # Highway 公式：gate * H(x) + (1-gate) * x
            x = gate * F.relu(transform) + (1 - gate) * x
        return x

## 4) CharCNN 字符级编码器
这里把每个词拆成字符序列后，用多尺度卷积提取形态特征（如前后缀、词形变化）。
卷积结果经 max-pooling、Highway、线性投影后得到词向量，
是 ELMo “从字符构建词表示” 的核心模块。

In [5]:
# CharCNN（Code）
# 字符级 CNN：输入 (B,S,C) 的字符 id，输出 (B,S,word_dim) 的词表示
class CharCNN(nn.Module):
    # char_vocab_size: 字符表大小
    # char_embed_dim: 字符向量维度
    # word_dim: 输出词向量维度（供后续 BiLM 使用）
    def __init__(self, char_vocab_size, char_embed_dim=50, word_dim=256):
        super().__init__()
        # 字符嵌入层，padding_idx=0 表示 PAD 字符的向量不参与训练更新
        self.embed = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        
        # 多尺度卷积核大小（1~7）
        filter_sizes = [1, 2, 3, 4, 5, 6, 7]
        # 每种卷积核对应的通道数，总和约 2000（接近原论文实践）
        num_filters = [100, 100, 200, 200, 300, 400, 700]  # sum=2000
        # 构建 1D 卷积层列表：输入通道=char_embed_dim，输出通道=nf
        self.convs = nn.ModuleList([
            nn.Conv1d(char_embed_dim, nf, ks, padding=ks//2)
            for ks, nf in zip(filter_sizes, num_filters)
        ])
        
        # 卷积拼接后先过 Highway 再投影到 word_dim
        self.highway = Highway(sum(num_filters))
        self.proj = nn.Linear(sum(num_filters), word_dim)
        # 轻量 dropout，减少过拟合
        self.dropout = nn.Dropout(0.1)
    
    # char_ids 形状：(B, S, C)
    def forward(self, char_ids):
        B, S, C = char_ids.shape  # B: batch，S: 句长，C: 词内字符数
        x = self.embed(char_ids)  # -> (B, S, C, E)
        # 变形为卷积可用格式：(B*S, E, C)
        x = x.view(B * S, C, -1).transpose(1, 2)
        
        conv_outs = []  # 保存每个卷积核分支的池化结果
        for conv in self.convs:
            c = conv(x)  # -> (B*S, nf, C')
            # 对时间维（字符维）做全局最大池化，得到固定长度向量
            c = F.max_pool1d(c, c.shape[2]).squeeze(2)  # -> (B*S, nf)
            conv_outs.append(c)
        
        # 拼接所有卷积分支特征
        x = torch.cat(conv_outs, dim=1)  # -> (B*S, total_filters)
        # Highway 融合
        x = self.highway(x)
        # 线性投影到 word_dim
        x = self.proj(x)
        # dropout 正则
        x = self.dropout(x)
        # 还原回 (B, S, word_dim)
        return x.view(B, S, -1)

## 5) 双向语言模型（BiLM）
这一部分是 ELMo 的主干：堆叠双向 LSTM 并进行语言模型训练。
训练时同时计算前向 LM 损失和反向 LM 损失，再做平均。
此外还定义了 `layer_weights` 与 `gamma`，用于后续 ELMo 的标量混合。

In [6]:
# BiLM 模型（完整双向 + scalar mixing 准备）（Code）
# BiLM = 双向语言模型主干：字符编码 + 多层双向 LSTM + 语言模型头
class BiLM(nn.Module):
    # char_vocab_size: 字符词表大小
    # word_vocab_size: 词表大小（LM 预测类别数）
    # word_dim: 每层输出词向量维度
    # lstm_hidden: LSTM 单方向隐藏维度（双向后为 2*lstm_hidden）
    # num_layers: LSTM 层数
    def __init__(self, char_vocab_size, word_vocab_size, word_dim=256, lstm_hidden=512, num_layers=2):
        super().__init__()
        # 底层字符编码器
        self.char_cnn = CharCNN(char_vocab_size, word_dim=word_dim)
        self.word_dim = word_dim
        self.lstm_hidden = lstm_hidden
        self.num_layers = num_layers
        
        # 按层构建 LSTM 与投影层
        self.lstm_layers = nn.ModuleList()
        self.proj_layers = nn.ModuleList()
        input_size = word_dim  # 第 1 层输入来自 CharCNN
        for _ in range(num_layers):
            # 每层使用单层双向 LSTM
            lstm = nn.LSTM(input_size, lstm_hidden, num_layers=1, bidirectional=True, batch_first=True, dropout=0.0)
            self.lstm_layers.append(lstm)
            # 将双向输出投影回 word_dim，便于层间统一维度
            self.proj_layers.append(nn.Linear(lstm_hidden * 2, word_dim))
            input_size = word_dim  # 下一层输入维度固定为 word_dim
        
        # 语言模型分类头：共享用于前向与反向分支
        self.lm_head = nn.Linear(word_dim, word_vocab_size)
        
        # ELMo 标量混合参数：num_layers+1（包含 char_cnn 层）
        self.layer_weights = nn.Parameter(torch.zeros(num_layers + 1))
        # 缩放参数 gamma
        self.gamma = nn.Parameter(torch.ones(1))
    
    # 前向：
    # - 推理时 word_ids=None，返回所有层表示
    # - 训练时提供 word_ids，返回 biLM loss 与层表示
    def forward(self, char_ids, word_ids=None):
        # 第 0 层表示：CharCNN 输出
        reps = [self.char_cnn(char_ids)]  # 每个元素形状均为 (B, S, word_dim)
        
        h = reps[0]  # 当前层输入
        for i, (lstm, proj) in enumerate(zip(self.lstm_layers, self.proj_layers)):
            # 从第 2 层开始加入残差（示意性实现）
            if i > 0:
                h = h + reps[1]
            # 双向 LSTM 编码
            out, _ = lstm(h)  # out: (B, S, 2*lstm_hidden)
            # 投影回 word_dim
            proj_out = proj(out)  # (B, S, word_dim)
            reps.append(proj_out)
            h = proj_out
        
        # 使用最顶层表示做 LM 词预测
        logits = self.lm_head(reps[-1])  # (B, S, V)
        
        # 若无标签，直接返回各层表示供下游使用
        if word_ids is None:
            return reps
        
        # -------------------------
        # 前向语言模型损失（预测下一个词）
        # -------------------------
        shift_logits = logits[:, :-1, :]  # 去掉最后一个位置（无 next token）
        shift_targets = word_ids[:, 1:]   # 目标从第 2 个词开始
        loss_fwd = F.cross_entropy(
            shift_logits.reshape(-1, logits.size(-1)),
            shift_targets.reshape(-1),
            ignore_index=0  # 忽略 PAD
        )
        
        # -------------------------
        # 反向语言模型损失（通过反转序列近似实现）
        # -------------------------
        rev_char = char_ids.flip(1)  # 句长维反转
        rev_reps = [self.char_cnn(rev_char)]
        h = rev_reps[0]
        for i, (lstm, proj) in enumerate(zip(self.lstm_layers, self.proj_layers)):
            if i > 0:
                h = h + rev_reps[1]
            out, _ = lstm(h)
            proj_out = proj(out)
            rev_reps.append(proj_out)
            h = proj_out

        rev_logits = self.lm_head(rev_reps[-1])
        rev_shift_logits = rev_logits[:, 1:, :]        # 对齐“前一个词”预测
        rev_shift_targets = word_ids.flip(1)[:, 1:]    # 反转后的目标
        loss_bwd = F.cross_entropy(
            rev_shift_logits.reshape(-1, logits.size(-1)),
            rev_shift_targets.reshape(-1),
            ignore_index=0
        )
        
        # 双向损失取平均
        total_loss = (loss_fwd + loss_bwd) / 2
        return total_loss, reps

## 6) ELMo Embedding 封装
此处将 BiLM 各层表示做可学习的标量加权（scalar mixing），
得到可直接用于下游任务（如分类、NER、QA）的上下文词向量。

In [7]:
# ELMo Embedder（预测用，可直接用于下游任务）（Code）
# ELMo 封装：把 BiLM 的多层表示做可学习加权融合
class ELMo(nn.Module):
    # bilm: 已训练（或待训练）的 BiLM 主干模型
    def __init__(self, bilm):
        super().__init__()
        self.bilm = bilm  # 直接复用 bilm 内部的 layer_weights 与 gamma
    
    # 输入字符 id，输出上下文词向量
    def forward(self, char_ids):
        # 获取每一层表示（list，长度 = num_layers + 1）
        layer_reps = self.bilm(char_ids)
        # 对层权重做 softmax，确保权重和为 1
        softmax_weights = F.softmax(self.bilm.layer_weights, dim=0)
        # 标量混合：gamma * Σ(w_l * h_l)
        elmo = self.bilm.gamma * sum(w * r for w, r in zip(softmax_weights, layer_reps))
        return elmo  # 形状：(B, S, word_dim)

## 7) 训练循环与模型保存
这一部分定义训练与验证流程，包括优化器、学习率调度、梯度裁剪。
同时根据验证集困惑度（Perplexity）保存最佳模型参数。

In [8]:
# 训练函数（Code）
# 定义训练主函数：包含训练、验证、学习率调度、最佳模型保存
def train_elmo(model, train_loader, valid_loader, epochs=5, lr=1e-3):
    # AdamW：对 Transformer/LM 任务常用，weight_decay 有助于正则化
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    # 余弦退火学习率调度器：随 epoch 平滑衰减学习率
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_ppl = float('inf')  # 记录当前最优验证困惑度（越低越好）
    for epoch in range(epochs):
        # -------------------------
        # 1) 训练阶段
        # -------------------------
        model.train()  # 启用训练模式（如 dropout 生效）
        total_loss = 0
        for word_ids, char_ids in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            # 将 batch 数据移动到训练设备（GPU/CPU）
            word_ids = word_ids.to(device)
            char_ids = char_ids.to(device)
            
            optimizer.zero_grad()  # 清空旧梯度
            loss, _ = model(char_ids, word_ids)  # 前向并计算 biLM loss
            loss.backward()  # 反向传播
            # 梯度裁剪：防止 RNN/LSTM 训练时梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()  # 更新参数
            total_loss += loss.item()  # 累计 loss 便于统计
        
        # 计算并打印本轮平均训练损失
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")
        
        # -------------------------
        # 2) 验证阶段
        # -------------------------
        model.eval()  # 切换到评估模式
        val_loss = 0
        with torch.no_grad():  # 验证不需要梯度，节省显存
            for word_ids, char_ids in valid_loader:
                word_ids = word_ids.to(device)
                char_ids = char_ids.to(device)
                loss, _ = model(char_ids, word_ids)
                val_loss += loss.item()

        # 困惑度 Perplexity = exp(平均交叉熵)
        val_ppl = math.exp(val_loss / len(valid_loader))
        print(f"Valid Perplexity: {val_ppl:.2f}")
        
        # 更新学习率
        scheduler.step()
        
        # 若验证指标更优，保存模型参数
        if val_ppl < best_ppl:
            best_ppl = val_ppl
            torch.save(model.state_dict(), "elmo_best.pth")
            print("Saved best model")

# -------------------------
# 实例化数据集与 DataLoader
# -------------------------
train_ds = ELMoDataset(train_sents, word2idx, char2idx)
valid_ds = ELMoDataset(valid_sents, word2idx, char2idx)

# num_workers 可按本机 CPU 核心数调整；pin_memory 对 GPU 训练常有帮助
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

# -------------------------
# 实例化模型并训练
# -------------------------
bilm = BiLM(CHAR_VOCAB_SIZE, VOCAB_SIZE, word_dim=256, lstm_hidden=512).to(device)
print("Model parameters:", sum(p.numel() for p in bilm.parameters()) / 1e6, "M")

# 开始训练（建议使用 GPU 环境）
train_elmo(bilm, train_loader, valid_loader, epochs=5)

Model parameters: 31.6439 M


Epoch 1: 100%|██████████| 743/743 [01:17<00:00,  9.59it/s]

Epoch 1 Train Loss: 6.8336


Valid Perplexity: 654.55
Saved best model


Epoch 2: 100%|██████████| 743/743 [01:17<00:00,  9.55it/s]

...

Epoch 4 Train Loss: 6.5317


Valid Perplexity: 654.37


Epoch 5: 100%|██████████| 743/743 [01:17<00:00,  9.55it/s]

Epoch 5 Train Loss: 6.5008


Valid Perplexity: 656.10


## 8) 推理示例：提取上下文词向量
最后用一句示例文本演示如何构造字符输入并调用 ELMo。
你会看到输出 embedding 的形状，便于接入下游网络。

In [9]:
# 预测 / 使用 ELMo embeddings（Code）
# 1) 加载训练阶段保存的最佳参数
bilm.load_state_dict(torch.load("elmo_best.pth", map_location=device))
# 2) 构建 ELMo 封装并切换到评估模式
elmo_model = ELMo(bilm).to(device)
elmo_model.eval()

# 3) 准备一条测试句子（示例）
test_sent = "the quick brown fox jumps over the lazy dog".split()
# 词级 id（本单元主要展示流程，实际这里只用于参考）
test_word_ids = [word2idx.get(w, word2idx["<UNK>"]) for w in test_sent]

# 4) 构造字符级输入：每个词 -> 固定长度字符 id 序列
test_char_ids_list = []
for w in test_sent:
    # 超过 MAX_WORD_LEN 的字符截断，未知字符映射为 <UNK>
    chs = [char2idx.get(c, char2idx["<UNK>"]) for c in w[:MAX_WORD_LEN]]
    # 不足部分补 PAD
    chs += [char2idx["<PAD>"]] * (MAX_WORD_LEN - len(chs))
    test_char_ids_list.append(chs)

# 增加 batch 维度并移动到设备：形状 (1, S, C)
test_char_ids = torch.tensor([test_char_ids_list]).to(device)

# 5) 前向推理：提取上下文词向量
with torch.no_grad():
    embeddings = elmo_model(test_char_ids)  # 输出形状 (1, S, 256)

# 6) 打印结果信息
print("ELMo embeddings shape:", embeddings.shape)
print("Ready for downstream tasks (e.g. classification, NER, etc.)")

ELMo embeddings shape: torch.Size([1, 9, 256])
Ready for downstream tasks (e.g. classification, NER, etc.)
